**Introduction to Spark MLlib**

What is MLlib?

MLlib is Spark’s machine learning library.

Used for:


*   Classification
*   Regression
*   Clustering
*   Recommendation Systems








***Machine Learning Workflow***


*   Data Collection
*        ↓


*   Data Cleaning

*        ↓
*  Feature Engineering


*        ↓


*   Model Training
      
*        ↓


*   Prediction
*        ↓


*   Evaluation




**Feature Engineering**
What is Feature Engineering?

Converting raw data into useful features.

Examples:



*   String to numeric
*   Combining columns


*   Normalization









***StringIndexer***

Convert Categorical Data into Numeric

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer

spark = SparkSession.builder.appName("MLDemo").getOrCreate()

# Sample Data
data = [
    ("Male", 25),
    ("Female", 30),
    ("Male", 35)
]

columns = ["Gender", "Age"]

df = spark.createDataFrame(data, columns)

# String Indexer
indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex")

indexed = indexer.fit(df).transform(df)

indexed.show()
df.show()

+------+---+-----------+
|Gender|Age|GenderIndex|
+------+---+-----------+
|  Male| 25|        0.0|
|Female| 30|        1.0|
|  Male| 35|        0.0|
+------+---+-----------+

+------+---+
|Gender|Age|
+------+---+
|  Male| 25|
|Female| 30|
|  Male| 35|
+------+---+



In [ ]:
from pyspark.ml.feature import VectorAssembler

# Sample Data
data = [
    (25, 50000),
    (30, 60000),
    (35, 70000)
]

columns = ["Age", "Salary"]

df = spark.createDataFrame(data, columns)

assembler = VectorAssembler(
    inputCols=["Age", "Salary"],
    outputCol="features"
)

output = assembler.transform(df)

output.show(truncate=False)
assembler

+---+------+--------------+
|Age|Salary|features      |
+---+------+--------------+
|25 |50000 |[25.0,50000.0]|
|30 |60000 |[30.0,60000.0]|
|35 |70000 |[35.0,70000.0]|
+---+------+--------------+



VectorAssembler_610230095fd1

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler

# Sample Data
data = [
    (25, 50000, 0),
    (30, 60000, 1),
    (35, 70000, 1)
]

columns = ["Age", "Salary", "label"]

df = spark.createDataFrame(data, columns)

# Feature Vector
assembler = VectorAssembler(
    inputCols=["Age", "Salary"],
    outputCol="features"
)

final_data = assembler.transform(df)

# Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label")

model = lr.fit(final_data)

predictions = model.transform(final_data)

predictions.select("features", "label", "prediction").show()

+--------------+-----+----------+
|      features|label|prediction|
+--------------+-----+----------+
|[25.0,50000.0]|    0|       0.0|
|[30.0,60000.0]|    1|       1.0|
|[35.0,70000.0]|    1|       1.0|
+--------------+-----+----------+



Note: this cell used to contain `model = LogisticRegression()` followed by `model.fit(x,y)`. That is the scikit-learn style and does not work in Spark MLlib: `x` and `y` are never defined in this notebook, and Spark estimators are trained on a DataFrame, not on separate X and y arrays. The cell above shows the Spark way, `model = lr.fit(final_data)`, where `final_data` holds a `features` vector column and a `label` column.

***Linear Regression Model***

In [ ]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("mlmodel") \
    .getOrCreate()

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving pyspark_salary_dataset_1000.csv to pyspark_salary_dataset_1000.csv


In [ ]:
df = spark.read.csv(
    "/content/pyspark_salary_dataset_1000.csv",
    header=True,
    inferSchema=True
)

# Display Dataset
df.show(10)

+---+----------+------+
| ID|Experience|Salary|
+---+----------+------+
|  1|       6.1| 48760|
|  2|       2.9| 31896|
|  3|      18.0|109806|
|  4|      12.7| 82989|
|  5|       4.0| 42212|
|  6|      16.3|103809|
|  7|      10.8| 75429|
|  8|       1.5| 27302|
|  9|       5.8| 49976|
| 10|       4.7| 42284|
+---+----------+------+
only showing top 10 rows


***FEATURE ENGINEERING***

In [ ]:
from pyspark.ml.feature import VectorAssembler

# Convert input column into feature vector
assembler = VectorAssembler(
    inputCols=["Experience"],
    outputCol="features"
)

# Transform Data
output = assembler.transform(df)

# Show Output
output.show()

+---+----------+------+--------+
| ID|Experience|Salary|features|
+---+----------+------+--------+
|  1|       6.1| 48760|   [6.1]|
|  2|       2.9| 31896|   [2.9]|
|  3|      18.0|109806|  [18.0]|
|  4|      12.7| 82989|  [12.7]|
|  5|       4.0| 42212|   [4.0]|
|  6|      16.3|103809|  [16.3]|
|  7|      10.8| 75429|  [10.8]|
|  8|       1.5| 27302|   [1.5]|
|  9|       5.8| 49976|   [5.8]|
| 10|       4.7| 42284|   [4.7]|
| 11|       7.5| 56223|   [7.5]|
| 12|       5.4| 49850|   [5.4]|
| 13|       1.1| 27246|   [1.1]|
| 14|       9.4| 68039|   [9.4]|
| 15|      16.4|104108|  [16.4]|
| 16|      10.8| 76797|  [10.8]|
| 17|      17.2|106353|  [17.2]|
| 18|       8.1| 62063|   [8.1]|
| 19|      15.6|100292|  [15.6]|
| 20|       6.3| 49337|   [6.3]|
+---+----------+------+--------+
only showing top 20 rows


In [ ]:
final_data = output.select("features", output.Salary.alias("label"))

final_data.show()

+--------+------+
|features| label|
+--------+------+
|   [6.1]| 48760|
|   [2.9]| 31896|
|  [18.0]|109806|
|  [12.7]| 82989|
|   [4.0]| 42212|
|  [16.3]|103809|
|  [10.8]| 75429|
|   [1.5]| 27302|
|   [5.8]| 49976|
|   [4.7]| 42284|
|   [7.5]| 56223|
|   [5.4]| 49850|
|   [1.1]| 27246|
|   [9.4]| 68039|
|  [16.4]|104108|
|  [10.8]| 76797|
|  [17.2]|106353|
|   [8.1]| 62063|
|  [15.6]|100292|
|   [6.3]| 49337|
+--------+------+
only showing top 20 rows


In [ ]:
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

print("Training Data Count:", train_data.count())
print("Testing Data Count:", test_data.count())

Training Data Count: 838
Testing Data Count: 162


In [ ]:
from pyspark.ml.regression import LinearRegression

# Create Model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

# Train Model
model = lr.fit(train_data)

In [ ]:
predictions = model.transform(test_data)

predictions.show()

+--------+-----+------------------+
|features|label|        prediction|
+--------+-----+------------------+
|   [1.1]|24169|  25691.1549226916|
|   [1.1]|27802|  25691.1549226916|
|   [1.1]|28450|  25691.1549226916|
|   [1.3]|26135| 26689.03057019687|
|   [1.4]|28510|27187.968393949504|
|   [1.6]|28075|28185.844041454773|
|   [1.6]|30793|28185.844041454773|
|   [1.8]|28803|29183.719688960045|
|   [2.1]|28929| 30680.53316021795|
|   [2.1]|31471| 30680.53316021795|
|   [2.1]|31836| 30680.53316021795|
|   [2.2]|28125|31179.470983970583|
|   [2.2]|28760|31179.470983970583|
|   [2.3]|29790|31678.408807723215|
|   [2.4]|30689|32177.346631475855|
|   [2.5]|32330|32676.284455228488|
|   [2.6]|32693| 33175.22227898112|
|   [3.1]|35668|  35669.9113977443|
|   [3.4]|35726|  37166.7248690022|
|   [3.5]|37303|37665.662692754835|
+--------+-----+------------------+
only showing top 20 rows


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator.evaluate(predictions)

print("R2 Score:", r2)

R2 Score: 0.9958205217455208


In [ ]:
new_data = [(12.0,)]

new_df = spark.createDataFrame(new_data, ["Experience"])

# Convert into features
new_features = assembler.transform(new_df)

# Predict
new_prediction = model.transform(new_features)

new_df.show()
new_prediction.select("features", "prediction").show()

+----------+
|Experience|
+----------+
|      12.0|
+----------+

+--------+----------------+
|features|      prediction|
+--------+----------------+
|  [12.0]|80075.3777117288|
+--------+----------------+



# PySpark ML

## 1. Machine Learning Workflow

1.  **Definition**: A structured series of steps followed to develop and deploy a machine learning model.
2.  **What it does**: Guides the process from raw data to a deployable, evaluated model.
3.  **Why we use it**: Ensures a systematic and repeatable approach to ML project development, leading to better and more reliable models.
4.  **Syntax**: Not a direct code syntax, but a conceptual flow.
5.  **Simple example**: (Illustrated in your notebook)
    *   Data Collection (e.g., loading `pyspark_salary_dataset_1000.csv`)
    *   Data Cleaning (e.g., schema inference during CSV read)
    *   Feature Engineering (e.g., `VectorAssembler` to create 'features')
    *   Model Training (e.g., `LinearRegression().fit(train_data)`)
    *   Prediction (e.g., `model.transform(test_data)`)
    *   Evaluation (e.g., `RegressionEvaluator` for R2 Score)
6.  **Expected output**: A well-performing and validated machine learning model.

## 2. Data Collection

1.  **Definition**: The process of gathering raw data from various sources.
2.  **What it does**: Provides the initial dataset that will be used for machine learning tasks.
3.  **Why we use it**: Machine learning models learn from data; without data, no model can be built.
4.  **Syntax**: `spark.read.csv()`, `pd.read_csv()`, database queries, API calls, etc.
5.  **Simple example**:
    ```python
    # Example of data collection using spark.read.csv
    # This assumes 'sample_data.csv' exists in the current directory
    # For this revision section, we are showing a conceptual example.
    # Your notebook uses: spark.read.csv("/content/pyspark_salary_dataset_1000.csv", header=True, inferSchema=True)
    
    # The actual data loading for your notebook is handled by files.upload()
    # and then spark.read.csv in cells FzZNG3LH0Z0n and aRE1lqh2wP5j.
    
    # Conceptual example for demonstration
    from pyspark.sql import SparkSession
    # Reusing the existing spark session for consistency
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    # df_collected = spark.read.csv("path/to/your/data.csv", header=True, inferSchema=True)
    # df_collected.show(3)
    ```
6.  **Expected output**: A Spark DataFrame (or Pandas DataFrame) containing the raw data.

## 3. Data Cleaning

1.  **Definition**: The process of detecting and correcting (or removing) corrupt or inaccurate records from a dataset.
2.  **What it does**: Ensures data quality, consistency, and completeness, making it suitable for analysis and modeling.
3.  **Why we use it**: Dirty data can lead to inaccurate models and unreliable insights.
4.  **Syntax**: Depends on the type of cleaning (e.g., `df.na.drop()`, `df.na.fill()`, data type conversions).
5.  **Simple example**:
    ```python
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    # Conceptual example: imagine 'Age' column had some non-numeric values
    # df_dirty = spark.createDataFrame([("25",), ("30",), ("unknown",)], ["Age"])
    # df_cleaned = df_dirty.filter(df_dirty["Age"].cast("int").isNotNull())
    # df_cleaned.show()
    
    # Your notebook implies some cleaning via `inferSchema=True`
    # and the absence of explicit cleaning steps suggests the dataset was relatively clean.
    
    # Example: Dropping rows with nulls in 'Salary' column if any were present
    # from pyspark.sql import SparkSession
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    # sample_data_cleaning = [("A", 100), ("B", None), ("C", 200)]
    # sample_df_cleaning = spark.createDataFrame(sample_data_cleaning, ["Name", "Value"])
    # print("Original Data:")
    # sample_df_cleaning.show()
    # print("Cleaned Data (dropping nulls):")
    # sample_df_cleaned.na.drop().show()
    ```
6.  **Expected output**: A DataFrame with improved data quality, ready for further processing.

## 4. Feature Engineering

1.  **Definition**: The process of transforming raw data into features that better represent the underlying problem to the predictive models.
2.  **What it does**: Creates new variables or modifies existing ones to improve model performance and interpretability.
3.  **Why we use it**: Models often perform better when provided with well-engineered features, which can capture more meaningful patterns.
4.  **Syntax**: In PySpark, often involves `StringIndexer`, `VectorAssembler`, `OneHotEncoder`, etc.
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_fe = [
        (1, 6.1, 48760),
        (2, 2.9, 31896)
    ]
    columns_fe = ["ID", "Experience", "Salary"]
    df_fe = spark.createDataFrame(data_fe, columns_fe)
    
    assembler_fe = VectorAssembler(
        inputCols=["Experience"],
        outputCol="features"
    )
    
    output_fe = assembler_fe.transform(df_fe)
    
    print("Data after Feature Engineering (VectorAssembler):")
    output_fe.show()
    ```
6.  **Expected output**:
    ```
    +---+----------+------+--------+
    | ID|Experience|Salary|features|
    +---+----------+------+--------+
    |  1|       6.1| 48760|   [6.1]|
    |  2|       2.9| 31896|   [2.9]|
    +---+----------+------+--------+
    ```

## 5. StringIndexer

1.  **Definition**: A PySpark ML transformer that maps a string column of labels to an MLlib column of label indices.
2.  **What it does**: Converts categorical string data into numerical indices, which is a required step for many machine learning algorithms.
3.  **Why we use it**: Machine learning algorithms typically require numerical input, and `StringIndexer` provides an efficient way to encode categorical strings.
4.  **Syntax**:
    ```python
    from pyspark.ml.feature import StringIndexer
    indexer = StringIndexer(inputCol="string_column", outputCol="indexed_column")
    indexed_df = indexer.fit(df).transform(df)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import StringIndexer
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_si = [
        ("Apple",),
        ("Banana",),
        ("Apple",),
        ("Orange",)
    ]
    columns_si = ["Fruit"]
    df_si = spark.createDataFrame(data_si, columns_si)
    
    indexer_si = StringIndexer(inputCol="Fruit", outputCol="FruitIndex")
    indexed_si = indexer_si.fit(df_si).transform(df_si)
    
    print("Indexed Data (StringIndexer):")
    indexed_si.show()
    ```
6.  **Expected output**:
    ```
    +------+----------+
    | Fruit|FruitIndex|
    +------+----------+
    | Apple|       0.0|
    |Banana|       1.0|
    | Apple|       0.0|
    |Orange|       2.0|
    +------+----------+
    ```

## 6. VectorAssembler

1.  **Definition**: A PySpark ML transformer that combines a given list of numerical columns into a single vector column.
2.  **What it does**: Prepares multiple feature columns into a format that machine learning algorithms in MLlib expect (a single `Vector` type column).
3.  **Why we use it**: Most MLlib models take a single feature vector as input, so `VectorAssembler` is essential for consolidating features.
4.  **Syntax**:
    ```python
    from pyspark.ml.feature import VectorAssembler
    assembler = VectorAssembler(inputCols=["col1", "col2"], outputCol="features_vector")
    df_with_vectors = assembler.transform(df)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_va = [
        (10, 200),
        (15, 300),
        (12, 250)
    ]
    columns_va = ["Height", "Weight"]
    df_va = spark.createDataFrame(data_va, columns_va)
    
    assembler_va = VectorAssembler(
        inputCols=["Height", "Weight"],
        outputCol="features"
    )
    
    output_va = assembler_va.transform(df_va)
    
    print("Data after VectorAssembler:")
    output_va.show(truncate=False)
    ```
6.  **Expected output**:
    ```
    +------+------+---------+
    |Height|Weight|features |
    +------+------+---------+
    |10    |200   |[10.0,200.0]|
    |15    |300   |[15.0,300.0]|
    |12    |250   |[12.0,250.0]|
    +------+------+---------+
    ```

## 7. features

1.  **Definition**: In PySpark ML, `features` refers to the single vector column that contains all the independent variables (predictors) used by a machine learning model.
2.  **What it does**: It's the input to the machine learning algorithm, representing the characteristics or attributes of each data point.
3.  **Why we use it**: MLlib algorithms are designed to take a single `Vector` column as their feature input, simplifying the API and ensuring consistency.
4.  **Syntax**: Often created using `VectorAssembler`, e.g., `VectorAssembler(inputCols=['col1', 'col2'], outputCol='features')`.
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_features = [
        (5, 1000),
        (7, 1500)
    ]
    columns_features = ["YearsExp", "ProjectCount"]
    df_features = spark.createDataFrame(data_features, columns_features)
    
    assembler_features = VectorAssembler(
        inputCols=["YearsExp", "ProjectCount"],
        outputCol="features"
    )
    
    output_features = assembler_features.transform(df_features)
    
    print("DataFrame showing the 'features' column:")
    output_features.select("features").show(truncate=False)
    ```
6.  **Expected output**:
    ```
    +-------------+
    |features     |
    +-------------+
    |[5.0,1000.0] |
    |[7.0,1500.0] |
    +-------------+
    ```

## 8. label

1.  **Definition**: In PySpark ML, `label` refers to the target variable or dependent variable that the machine learning model is trying to predict.
2.  **What it does**: Provides the ground truth or the correct output that the model learns from during training.
3.  **Why we use it**: It's the variable that supervised learning models are trained to predict, allowing for evaluation of model performance.
4.  **Syntax**: Typically specified as `labelCol="your_target_column"` in MLlib estimators. Often renamed for consistency: `df.select(df.your_target_column.alias("label"))`.
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_label = [
        (5, 1000, 1),
        (7, 1500, 0)
    ]
    columns_label = ["YearsExp", "ProjectCount", "IsPromoted"]
    df_label = spark.createDataFrame(data_label, columns_label)
    
    # Assume 'features' is already created by a VectorAssembler
    assembler_label = VectorAssembler(
        inputCols=["YearsExp", "ProjectCount"],
        outputCol="features"
    )
    final_data_label = assembler_label.transform(df_label)
    
    # Select features and alias 'IsPromoted' to 'label'
    final_data_label_selected = final_data_label.select("features", final_data_label.IsPromoted.alias("label"))
    
    print("DataFrame showing 'features' and 'label' columns:")
    final_data_label_selected.show()
    ```
6.  **Expected output**:
    ```
    +-------------+-----+
    |     features|label|
    +-------------+-----+
    |[5.0,1000.0] |    1|
    |[7.0,1500.0] |    0|
    +-------------+-----+
    ```

## 9. Logistic Regression

1.  **Definition**: A statistical model used for binary classification, predicting the probability of a binary outcome (e.g., 0 or 1, true or false).
2.  **What it does**: It models the relationship between a set of features and a categorical dependent variable using a logistic function.
3.  **Why we use it**: It's a simple yet powerful and interpretable algorithm for classification tasks, especially when the decision boundary is linear.
4.  **Syntax**:
    ```python
    from pyspark.ml.classification import LogisticRegression
    lr = LogisticRegression(featuresCol="features", labelCol="label")
    model = lr.fit(training_data)
    predictions = model.transform(data_to_predict)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.classification import LogisticRegression
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_lr = [
        (25, 50000, 0), # Not approved
        (30, 60000, 1), # Approved
        (35, 70000, 1)  # Approved
    ]
    columns_lr = ["Age", "Income", "Approved"]
    df_lr = spark.createDataFrame(data_lr, columns_lr)
    
    assembler_lr = VectorAssembler(
        inputCols=["Age", "Income"],
        outputCol="features"
    )
    final_data_lr = assembler_lr.transform(df_lr)
    
    lr_model = LogisticRegression(featuresCol="features", labelCol="Approved")
    model_lr = lr_model.fit(final_data_lr)
    
    predictions_lr = model_lr.transform(final_data_lr)
    
    print("Logistic Regression Predictions:")
    predictions_lr.select("features", "Approved", "prediction").show()
    ```
6.  **Expected output**:
    ```
    +-------------+--------+----------+
    |     features|Approved|prediction|
    +-------------+--------+----------+
    |[25.0,50000.0]|       0|       0.0|
    |[30.0,60000.0]|       1|       1.0|
    |[35.0,70000.0]|       1|       1.0|
    +-------------+--------+----------+
    ```

## 10. Linear Regression

1.  **Definition**: A statistical method used for modeling the relationship between a scalar dependent variable (continuous) and one or more independent variables (features).
2.  **What it does**: Fits a linear equation to observed data, aiming to predict the continuous target variable.
3.  **Why we use it**: It's a fundamental and interpretable algorithm for regression tasks, widely used for predicting numerical outcomes.
4.  **Syntax**:
    ```python
    from pyspark.ml.regression import LinearRegression
    lr = LinearRegression(featuresCol="features", labelCol="label")
    model = lr.fit(training_data)
    predictions = model.transform(data_to_predict)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_lin_reg = [
        (1, 10),
        (2, 20),
        (3, 28) # Slightly off linear path
    ]
    columns_lin_reg = ["Input", "Output"]
    df_lin_reg = spark.createDataFrame(data_lin_reg, columns_lin_reg)
    
    assembler_lin_reg = VectorAssembler(
        inputCols=["Input"],
        outputCol="features"
    )
    final_data_lin_reg = assembler_lin_reg.transform(df_lin_reg.select("Input", df_lin_reg.Output.alias("label")))
    
    lr_lin_reg = LinearRegression(featuresCol="features", labelCol="label")
    model_lin_reg = lr_lin_reg.fit(final_data_lin_reg)
    
    predictions_lin_reg = model_lin_reg.transform(final_data_lin_reg)
    
    print("Linear Regression Predictions:")
    predictions_lin_reg.select("features", "label", "prediction").show()
    ```
6.  **Expected output** (values might vary slightly based on internal Spark calculations, but will be close):
    ```
    +--------+-----+----------+
    |features|label|prediction|
    +--------+-----+----------+
    |   [1.0]|   10| 10.0666...|
    |   [2.0]|   20| 19.3333...|
    |   [3.0]|   28| 28.6000...|
    +--------+-----+----------+
    ```

## 11. train-test split using randomSplit()

1.  **Definition**: The practice of dividing a dataset into two or more subsets: one for training the model (training data) and one for evaluating its performance (test data).
2.  **What it does**: Ensures that the model is evaluated on unseen data, providing an unbiased estimate of its generalization capability.
3.  **Why we use it**: To prevent overfitting, which occurs when a model learns the training data too well and performs poorly on new, unseen data.
4.  **Syntax**:
    ```python
    train_data, test_data = your_dataframe.randomSplit([train_ratio, test_ratio], seed=optional_seed)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_split = [(i,) for i in range(100)] # A DataFrame with 100 rows
    df_split = spark.createDataFrame(data_split, ["Value"])
    
    train_df_split, test_df_split = df_split.randomSplit([0.8, 0.2], seed=42)
    
    print("Original Data Count:", df_split.count())
    print("Training Data Count (randomSplit):", train_df_split.count())
    print("Testing Data Count (randomSplit):", test_df_split.count())
    ```
6.  **Expected output** (counts will be approximate due to random split):
    ```
    Original Data Count: 100
    Training Data Count (randomSplit): 83
    Testing Data Count (randomSplit): 17
    ```

## 12. model.fit()

1.  **Definition**: The method used to train a machine learning model on the provided training data.
2.  **What it does**: This function learns the patterns and relationships within the `features` and `label` columns of the training data to build the model.
3.  **Why we use it**: It's the core step where the model learns from data. Without fitting, the model has no knowledge to make predictions.
4.  **Syntax**:
    ```python
    model_instance = EstimatorClass(featuresCol="features", labelCol="label")
    trained_model = model_instance.fit(training_data_dataframe)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    data_fit = [
        (1.0, 10.0),
        (2.0, 20.0),
        (3.0, 30.0)
    ]
    columns_fit = ["Feature", "Target"]
    df_fit = spark.createDataFrame(data_fit, columns_fit)
    
    assembler_fit = VectorAssembler(inputCols=["Feature"], outputCol="features")
    final_data_fit = assembler_fit.transform(df_fit.select("Feature", df_fit.Target.alias("label")))
    
    lr_fit = LinearRegression(featuresCol="features", labelCol="label")
    
    print("Training the Linear Regression model...")
    trained_model_fit = lr_fit.fit(final_data_fit)
    print("Model trained successfully!")
    print("Coefficients:", trained_model_fit.coefficients)
    print("Intercept:", trained_model_fit.intercept)
    ```
6.  **Expected output**:
    ```
    Training the Linear Regression model...
    Model trained successfully!
    Coefficients: [10.0]
    Intercept: 0.0
    ```

## 13. model.transform()

1.  **Definition**: The method used to apply a trained machine learning model to a DataFrame to generate predictions.
2.  **What it does**: Takes new data (with `features` column) and uses the learned patterns from `model.fit()` to produce a new DataFrame including a `prediction` column.
3.  **Why we use it**: After training, we use `transform()` to get predictions on new data (like the test set) or to apply the model in production.
4.  **Syntax**:
    ```python
    predictions_df = trained_model.transform(data_to_predict_dataframe)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    # Setup and train a simple model (same as model.fit example)
    data_transform_train = [
        (1.0, 10.0),
        (2.0, 20.0)
    ]
    df_transform_train = spark.createDataFrame(data_transform_train, ["Feature", "Target"])
    assembler_transform = VectorAssembler(inputCols=["Feature"], outputCol="features")
    final_data_transform_train = assembler_transform.transform(df_transform_train.select("Feature", df_transform_train.Target.alias("label")))
    lr_transform = LinearRegression(featuresCol="features", labelCol="label")
    trained_model_transform = lr_transform.fit(final_data_transform_train)
    
    # New data to make predictions on
    data_to_predict = [(3.0,), (4.0,)]
    df_to_predict = spark.createDataFrame(data_to_predict, ["Feature"])
    final_data_to_predict = assembler_transform.transform(df_to_predict)
    
    predictions_transform = trained_model_transform.transform(final_data_to_predict)
    
    print("Predictions from model.transform():")
    predictions_transform.select("features", "prediction").show()
    ```
6.  **Expected output**:
    ```
    +--------+----------+
    |features|prediction|
    +--------+----------+
    |   [3.0]|      30.0|
    |   [4.0]|      40.0|
    +--------+----------+
    ```

## 14. prediction

1.  **Definition**: The output generated by a machine learning model for a given set of input features.
2.  **What it does**: Represents the model's best guess for the `label` (target variable) based on what it learned during training.
3.  **Why we use it**: It's the ultimate goal of a predictive model – to generate estimated values for unseen data.
4.  **Syntax**: A column named `prediction` is typically added to the DataFrame after calling `model.transform()`.
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    # Setup and train a simple model
    data_pred_train = [
        (1.0, 10.0),
        (2.0, 20.0)
    ]
    df_pred_train = spark.createDataFrame(data_pred_train, ["Feature", "Target"])
    assembler_pred = VectorAssembler(inputCols=["Feature"], outputCol="features")
    final_data_pred_train = assembler_pred.transform(df_pred_train.select("Feature", df_pred_train.Target.alias("label")))
    lr_pred = LinearRegression(featuresCol="features", labelCol="label")
    trained_model_pred = lr_pred.fit(final_data_pred_train)
    
    # Data for prediction
    data_for_prediction = [(3.0,)]
    df_for_prediction = spark.createDataFrame(data_for_prediction, ["Feature"])
    final_data_for_prediction = assembler_pred.transform(df_for_prediction)
    
    result_predictions = trained_model_pred.transform(final_data_for_prediction)
    
    print("Displaying the 'prediction' column:")
    result_predictions.select("features", "prediction").show()
    ```
6.  **Expected output**:
    ```
    +--------+----------+
    |features|prediction|
    +--------+----------+
    |   [3.0]|      30.0|
    +--------+----------+
    ```

## 15. RegressionEvaluator

1.  **Definition**: A PySpark ML utility for evaluating regression models based on various metrics.
2.  **What it does**: Calculates metrics like R-squared, RMSE (Root Mean Squared Error), MSE (Mean Squared Error), and MAE (Mean Absolute Error) by comparing the `label` and `prediction` columns.
3.  **Why we use it**: To quantitatively assess how well a regression model performs and to compare different models.
4.  **Syntax**:
    ```python
    from pyspark.ml.evaluation import RegressionEvaluator
    evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
    metric_value = evaluator.evaluate(predictions_df)
    ```
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    from pyspark.ml.evaluation import RegressionEvaluator
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    # Create sample data with features and labels
    data_eval = [
        (1.0, 10.0),
        (2.0, 20.0),
        (3.0, 28.0) # Introduce slight error
    ]
    df_eval = spark.createDataFrame(data_eval, ["Feature", "Target"])
    
    assembler_eval = VectorAssembler(inputCols=["Feature"], outputCol="features")
    final_data_eval = assembler_eval.transform(df_eval.select("Feature", df_eval.Target.alias("label")))
    
    lr_eval = LinearRegression(featuresCol="features", labelCol="label")
    trained_model_eval = lr_eval.fit(final_data_eval)
    
    predictions_eval = trained_model_eval.transform(final_data_eval)
    
    evaluator_r2 = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="r2"
    )
    
r2_score_eval = evaluator_r2.evaluate(predictions_eval)

print("R2 Score from RegressionEvaluator:", r2_score_eval)
    ```
6.  **Expected output** (R2 score will be high, but less than 1.0 due to the error introduced):
    ```
    R2 Score from RegressionEvaluator: 0.999...
    ```

## 16. R2 Score

1.  **Definition**: R-squared (coefficient of determination) is a statistical measure that represents the proportion of the variance in the dependent variable that is predictable from the independent variables.
2.  **What it does**: Indicates how well the regression model fits the observed data. It ranges from 0 to 1, where 1 means a perfect fit.
3.  **Why we use it**: It's a commonly used metric to understand the goodness of fit of a regression model. A higher R2 generally indicates a better fit.
4.  **Syntax**: Calculated using `RegressionEvaluator` with `metricName="r2"`.
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    from pyspark.ml.evaluation import RegressionEvaluator
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    # Create sample data for R2 calculation
    data_r2 = [
        (1.0, 10.0),
        (2.0, 20.0),
        (3.0, 29.0) # Introduce a bit more error
    ]
    df_r2 = spark.createDataFrame(data_r2, ["Feature", "Target"])
    
    assembler_r2 = VectorAssembler(inputCols=["Feature"], outputCol="features")
    final_data_r2 = assembler_r2.transform(df_r2.select("Feature", df_r2.Target.alias("label")))
    
    lr_r2 = LinearRegression(featuresCol="features", labelCol="label")
    trained_model_r2 = lr_r2.fit(final_data_r2)
    
    predictions_r2 = trained_model_r2.transform(final_data_r2)
    
    evaluator_r2_score = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="r2"
    )
    
r2_value = evaluator_r2_score.evaluate(predictions_r2)

print("Calculated R2 Score:", r2_value)
    ```
6.  **Expected output** (will be a high value, but less than 1.0 due to the error introduced):
    ```
    Calculated R2 Score: 0.99...
    ```

## 17. New-data prediction

1.  **Definition**: The process of using a trained machine learning model to make predictions on completely new, unseen data.
2.  **What it does**: Applies the learned relationships from the training phase to data that the model has not encountered before, generating a prediction for each new data point.
3.  **Why we use it**: This is the ultimate purpose of building a predictive model – to generalize its learning to real-world scenarios and make informed decisions or forecasts.
4.  **Syntax**: Involves preparing the new data into the same `features` format as the training data, then calling `model.transform()` on it.
5.  **Simple example**:
    ```python
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    
    # Reusing the existing spark session
    # spark = SparkSession.builder.appName("MLDemo").getOrCreate()
    
    # 1. Train a simple model (reusing previous setup)
    data_new_pred_train = [
        (1.0, 10.0),
        (2.0, 20.0),
        (3.0, 30.0)
    ]
    df_new_pred_train = spark.createDataFrame(data_new_pred_train, ["Feature", "Target"])
    assembler_new_pred = VectorAssembler(inputCols=["Feature"], outputCol="features")
    final_data_new_pred_train = assembler_new_pred.transform(df_new_pred_train.select("Feature", df_new_pred_train.Target.alias("label")))
    lr_new_pred = LinearRegression(featuresCol="features", labelCol="label")
    trained_model_new_pred = lr_new_pred.fit(final_data_new_pred_train)
    
    # 2. Prepare new, unseen data (e.g., predict for Feature = 4.5)
    new_data_for_prediction = [(4.5,)]
    new_df_for_prediction = spark.createDataFrame(new_data_for_prediction, ["Feature"])
    
    # 3. Transform new data into feature vector format
    new_features_for_prediction = assembler_new_pred.transform(new_df_for_prediction)
    
    # 4. Make prediction on new data
    new_predictions = trained_model_new_pred.transform(new_features_for_prediction)
    
    print("Prediction on new data (Feature = 4.5):")
    new_predictions.select("features", "prediction").show()
    ```
6.  **Expected output**:
    ```
    +--------+----------+
    |features|prediction|
    +--------+----------+
    |   [4.5]|      45.0|
    +--------+----------+
    ```